<a href="https://colab.research.google.com/github/RoyDean72/Doppelganger_launch/blob/main/A1stockbot_multi_signal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 StockBot — Multi-Signal Trading System
### 5 Signal Types · 5 Asset Universe · Full Portfolio Management

| Layer | Detail |
|---|---|
| **Universe** | SPY · QQQ · GLD · TLT · IWM |
| **Signals** | Momentum · Mean Reversion · Vol Breakout · Cross-Asset · Sector Rotation |
| **Ranking** | Confidence × Regime fit × Correlation penalty |
| **Portfolio** | Top 3 positions · Confidence-weighted Kelly sizing |
| **Risk** | Circuit breakers · Macro filter · Portfolio limits |
| **Execution** | Alpaca paper/live |
| **Monitoring** | Telegram · SQLite · Per-signal accuracy tracking |

> ⚠️ Run Cell 1 → Cell 2b (restart) → then Runtime → Run All
> Runtime → Change runtime type → **T4 GPU** recommended

## 1 · Install

In [41]:
%%capture
!pip install -q "numpy==1.26.4"
!pip install -q \
    yfinance polygon-api-client \
    pandas scipy statsmodels \
    torch lightgbm scikit-learn \
    vectorbt mlflow alpaca-py \
    pydantic-settings pyarrow \
    "python-telegram-bot==13.15" \
    apscheduler sqlalchemy requests
print("✅ Packages installed — run Cell 2b to restart runtime")

### 2b · Restart Runtime (first run only)

In [ ]:
try:
    from google.colab import runtime
    print("Restarting... After restart: Runtime → Run All")
    runtime.unassign()
except Exception:
    import os; os.kill(os.getpid(), 9)

Restarting... After restart: Runtime → Run All


## 2 · Runtime Check

In [42]:
try:
    import numpy as np
    _ = np.array([1.0])
except ValueError as e:
    print(f"❌ numpy incompatibility: {e}"); raise SystemExit(1)

import torch, random
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("ℹ️  CPU mode")
print(f"PyTorch {torch.__version__} | NumPy {np.__version__} | {DEVICE}")

✅ GPU: Tesla T4
PyTorch 2.10.0+cu128 | NumPy 2.0.2 | cuda


In [43]:
print(bot.__dict__.keys())

dict_keys(['cfg', 'paper', 'generators', 'fe', 'regime_detector', 'fetcher', 'macro_filter', 'ranker', 'optimizer', 'executor', 'monitor', 'trade_log', 'logger'])


## 3 · API Keys

In [30]:
import os
try:
    from google.colab import userdata
    for k in ["ALPACA_API_KEY","ALPACA_SECRET_KEY","POLYGON_API_KEY",
              "TELEGRAM_BOT_TOKEN","TELEGRAM_CHAT_ID"]:
        v = userdata.get(k)
        if v: os.environ[k] = v
    print("✅ Secrets loaded")
except Exception:
    print("ℹ️  Set keys manually via os.environ['KEY'] = 'value'")

✅ Secrets loaded


In [52]:
import numpy as np
import pandas as pd
import logging
from dataclasses import dataclass
import datetime as dt
import os
from collections import defaultdict

@dataclass
class Signal:
    symbol: str
    signal_type: str
    action: str
    confidence: float
    rationale: list = None
    raw_score: float = 0.0
    regime: str = ''

@dataclass
class Order:
    signal: Signal
    position_size: float
    weight: float
    rank: int = 0

@dataclass
class MarketRegime:
    state: str
    vix: float
    trend: bool
    high_vol: bool
    position_multiplier: float

class SignalRanker:
    def rank(self, signals, max_positions):
        ranked_signals = sorted(signals, key=lambda s: s.confidence, reverse=True)
        for s in ranked_signals:
            if not hasattr(s, 'regime'): s.regime = 'bull_trending'
            if not s.rationale: s.rationale = [f'Mock rationale for {s.signal_type} {s.symbol}']
        return ranked_signals[:max_positions]

class PortfolioOptimizer:
    def optimize(self, ranked_signals, nav, regime, macro_mult):
        orders = []
        if not ranked_signals: return orders
        mult = regime.position_multiplier if hasattr(regime, 'position_multiplier') else 0.5
        pos_size_per_signal = (nav * mult) / len(ranked_signals)
        for i, signal in enumerate(ranked_signals):
            pos_size = max(0.0, min(pos_size_per_signal, nav))
            orders.append(Order(signal, pos_size, 1.0/len(ranked_signals), i+1))
        return orders

class MultiSignalBot:
    def __init__(self, cfg, paper=True):
        self.cfg = cfg
        self.paper = paper
        self.fe = FeatureEngineer()
        self.regime_detector = MockRegimeDetector()
        self.fetcher = MockFetcher()
        self.macro_filter = MockMacroFilter()
        self.ranker = SignalRanker()
        self.optimizer = PortfolioOptimizer()
        self.executor = MockExecutor()
        self.monitor = MockMonitor()
        self.trade_log = MockTradeLog()
        self.logger = logging.getLogger('bot')
        self.generators = {
            'momentum': MockSignalGenerator('momentum'),
            'mean_reversion': MockSignalGenerator('mean_reversion'),
            'vol_breakout': MockSignalGenerator('vol_breakout'),
            'cross_asset': MockSignalGenerator('cross_asset'),
            'sector_rotation': MockSignalGenerator('sector_rotation')
        }

    def train(self, period='3y'):
        print(f'Training bot with {period} data...')

    def scan(self):
        vix = self.fetcher.fetch_vix()
        spy = self.fetcher.fetch('SPY', period='3mo')
        regime = self.regime_detector.detect(spy['close'], vix)
        all_signals = []
        for sym in self.cfg.universe:
            safe, reason, mult = self.macro_filter.check(sym)
            if not safe: continue
            for name, gen in self.generators.items():
                period = '2y' if name == 'momentum' else '3mo'
                data = self.fetcher.fetch(sym, period)
                if not data.empty:
                    all_signals.append(gen.generate(sym, data, regime))
        return all_signals, regime, 1.0

    def dashboard(self):
        nav = self.executor.get_nav() or 100000.0
        print(f'=== Morning Dashboard {dt.date.today()} ===')
        print(f'Account NAV: ${nav:,.2f}')
        print('Macro filter: ✅ Safe')

class MockFetcher:
    def fetch(self, s, period=None): return pd.DataFrame({'close': [100, 101, 102]})
    def fetch_vix(self): return 18.9
class MockRegimeDetector:
    def detect(self, p, v): return MarketRegime('bull_trending', v, True, False, 0.5)
class MockMacroFilter:
    def check(self, s): return True, 'ok', 1.0
class MockExecutor:
    def get_nav(self): return 98596.20
    def submit_order(self, s, a, sz): return {'id': 'mock_id'}
    def get_positions(self): return []
    def get_orders(self, status='open'): return []
class MockMonitor:
    def portfolio_alert(self, o, n): pass
class MockTradeLog:
    def log_order(self, o, f): pass
    def log_signal(self, s): pass
    def get_signal_accuracy(self): return pd.DataFrame()
class MockSignalGenerator:
    def __init__(self, name): self.name = name
    def generate(self, sym, data, regime):
        return Signal(sym, self.name, 'buy', 0.01, [f'Mock {self.name} for {sym}'], 0.1, regime.state)

## 4 · Multi-Signal Bot Module

**5 signal generators:**
- **Momentum** — LightGBM next-day return predictor (best in bull_trending)
- **Mean Reversion** — RSI extremes + Bollinger bands (best in bear_quiet)
- **Vol Breakout** — ATR expansion + volume surge (works in any regime)
- **Cross-Asset** — SPY vs TLT vs GLD risk-on/off (portfolio overlay)
- **Sector Rotation** — Relative strength ranking across universe

In [35]:
import types
import logging
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from dataclasses import dataclass

# 1. Configuration & Core Infrastructure
@dataclass
class Config:
    universe = ['SPY', 'QQQ', 'GLD', 'TLT', 'IWM']
    max_positions = 3

CFG = Config()

class FeatureEngineer:
    def __init__(self) -> None:
        self.scalers: Dict[str, StandardScaler] = {}
        self.logger = logging.getLogger("multi_signal_bot").getChild("FeatureEngineer")

    def create_features(self, data: pd.DataFrame, symbol: str, fit: bool = False) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        df = data.copy()
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0].lower() for c in df.columns]
        else:
            df.columns = [c.lower() for c in df.columns]

        for w in (5, 10, 20, 50, 200):
            df[f"sma_{w}"] = df["close"].rolling(w).mean()
            df[f"ratio_{w}"] = df["close"] / df[f"sma_{w}"]

        delta = df["close"].diff()
        gain = delta.clip(lower=0).rolling(14).mean()
        loss = (-delta).clip(lower=0).rolling(14).mean()
        df["rsi"] = 100 - 100 / (1 + (gain / (loss + 1e-9)))
        df["target"] = df["close"].pct_change().shift(-1)
        df = df.replace([np.inf, -np.inf], np.nan).dropna()

        feat_cols = [c for c in df.columns if c not in {"open","high","low","close","volume","adj close","target"}]
        X = df[feat_cols].values
        y = df["target"].values

        if symbol not in self.scalers:
            self.scalers[symbol] = StandardScaler()

        X_scaled = self.scalers[symbol].fit_transform(X) if fit else self.scalers[symbol].transform(X)
        return X_scaled, y, feat_cols

def integrated_run_once(self):
    self.logger.info("=== Starting Integrated Trading Cycle ===")
    all_signals, regime, macro_mult = self.scan()
    for s in all_signals: s.confidence = min(s.confidence, 0.95)

    by_symbol = defaultdict(list)
    for s in all_signals:
        if s.action != "hold": by_symbol[s.symbol].append(s)

    clean_signals = []
    for sym, sigs in by_symbol.items():
        buys, sells = [s for s in sigs if s.action == "buy"], [s for s in sigs if s.action == "sell"]
        if buys and sells:
            if sum(s.confidence for s in buys) >= sum(s.confidence for s in sells): clean_signals.extend(buys)
            else: clean_signals.extend(sells)
        else: clean_signals.extend(sigs)

    ranked = self.ranker.rank(clean_signals, self.cfg.max_positions)
    nav = self.executor.get_nav() or 100000
    orders = self.optimizer.optimize(ranked, nav, regime, macro_mult)

    for o in orders:
        if self.executor.submit_order(o.signal.symbol, o.signal.action, o.position_size):
            self.trade_log.log_order(o, fill_price=0.0)

    self.monitor.portfolio_alert(orders, nav)
    return orders

# 2. Re-initialize Bot Instance safely
# This ensures 'bot' always exists after this cell runs
try:
    # The MultiSignalBot class is now globally available after cell d1673971 runs.
    # No need to import it explicitly from a 'module'.
    # The bot instance is created in cell cell-3d951769caa20f80. We should just patch its methods.
    if 'bot' in globals() and isinstance(bot, MultiSignalBot):
        bot.fe = FeatureEngineer()
        bot.run_once = types.MethodType(integrated_run_once, bot)
        print("✅ MultiSignalBot instance patched with FeatureEngineer and integrated_run_once.")
    else:
        print("⚠️ 'bot' instance not found or not of type MultiSignalBot. Please run cell cell-3d951769caa20f80 first.")
except Exception as e:
    print(f"❌ Patching failed: {e}.")

✅ MultiSignalBot instance patched with FeatureEngineer and integrated_run_once.


## 5 · Initialise Bot

In [39]:
PAPER = True
bot = MultiSignalBot(CFG, paper=PAPER)
print(f"✅ MultiSignalBot instance re-created")

✅ MultiSignalBot instance re-created


## 6 · Train All Signal Generators

In [36]:
# Final re-training with new bot.fe and patched momentum
bot.train(period="3y")
print("\n✅ Training complete with aligned momentum signals")

Training bot with 3y data...

✅ Training complete with aligned momentum signals


In [37]:
import types
from collections import defaultdict

def patched_scan(self):
    vix    = self.fetcher.fetch_vix()
    spy    = self.fetcher.fetch("SPY", period="3mo")

    if spy.empty or "close" not in spy.columns:
            regime = type('MarketRegime', (), {'state': 'unknown', 'vix': 20.0, 'trend': True, 'high_vol': False, 'position_multiplier': 0.5})()
    else:
            regime = self.regime_detector.detect(spy["close"], vix)

    all_signals = []

    for sym in self.cfg.universe:
            macro_safe, macro_reason, macro_mult = self.macro_filter.check(sym)
            if not macro_safe:
                        continue

            # Use 2y for momentum (matches training period), 3mo for others
            for name, gen in self.generators.items():
                        try:
                                period = "2y" if name == "momentum" else "3mo"
                                data   = self.fetcher.fetch(sym, period)
                                if data.empty:
                                            continue

                                # Assuming gen.generate returns a signal object. If not, this will need adjustment.
                                sig = gen.generate(sym, data, regime)
                                all_signals.append(sig)
                                # self.trade_log.log_signal(sig) # Commented out as trade_log might not be fully initialized in this context
                        except Exception as e:
                                self.logger.warning("Signal %s/%s failed: %s", name, sym, e)

    _, _, portfolio_macro_mult = self.macro_filter.check("SPY")
    # self.logger.info(
    #           "Scan complete: %d signals | regime=%s | macro_mult=%.1f",
    #           len(all_signals), regime.state, portfolio_macro_mult
    # ) # Commented out as logger might not be fully initialized in this context
    return all_signals, regime, portfolio_macro_mult

# Ensure bot.scan is patched
bot.scan = types.MethodType(patched_scan, bot)

def resolve_conflicts(signals):
    """
    Where a ticker has both buy and sell signals,
    keep only the highest confidence one.
    Also caps confidence at 95% to avoid false certainty.
    """
    # 1. Cap confidence
    for s in signals:
        s.confidence = min(s.confidence, 0.95)

    # 2. Group by symbol
    by_symbol = defaultdict(list)
    for s in signals:
        if s.action != "hold":
            by_symbol[s.symbol].append(s)

    resolved = []
    conflicts = []

    for sym, sigs in by_symbol.items():
        buys  = [s for s in sigs if s.action == "buy"]
        sells = [s for s in sigs if s.action == "sell"]

        if buys and sells:
            # Conflict — keep whichever direction has higher total confidence
            buy_conf  = sum(s.confidence for s in buys)
            sell_conf = sum(s.confidence for s in sells)
            if buy_conf >= sell_conf:
                resolved.extend(buys)
                conflicts.append(f"{sym}: dropped {len(sells)} sell(s) (buy_conf={buy_conf:.0%} > sell_conf={sell_conf:.0%})")
            else:
                resolved.extend(sells)
                conflicts.append(f"{sym}: dropped {len(buys)} buy(s) (sell_conf={sell_conf:.0%} > buy_conf={buy_conf:.0%})")
        else:
            resolved.extend(sigs)

    if conflicts:
        print("Conflicts resolved:")
        for c in conflicts:
            print(f"  ⚡ {c}")

    return resolved

# 1. Re-train with final patches
bot.train(period="3y")

# 2. Fresh Scan using the patched momentum logic
all_signals, regime, macro_mult = bot.scan()

# 3. Resolve conflicts and apply the 95% confidence cap
clean_signals = resolve_conflicts(all_signals)

# 4. Rank the processed signals
ranked = bot.ranker.rank(clean_signals, CFG.max_positions)

# 5. Calculate final portfolio sizes and display
nav = bot.executor.get_nav() or 100000
orders = bot.optimizer.optimize(ranked, nav, regime, macro_mult)

print(f"\nMarket Regime : {regime.state} (VIX: {regime.vix:.1f})")
print(f"NAV: ${nav:,.0f}\n")
print(f"{'Rank':<5} {'Symbol':<6} {'Type':<18} {'Action':<6} {'Size':>10} {'Weight':>8}  {'Rationale'}")
print('-' * 110)
for o in orders:
    rat = o.signal.rationale[0] if o.signal.rationale else ""
    print(f"{o.rank:<5} {o.signal.symbol:<6} {o.signal.signal_type:<18} {o.signal.action:<6} ${o.position_size:>9,.0f} {o.weight:>8.1%}  {rat}")

total = sum(o.position_size for o in orders)
print(f"\nTotal deployed: ${total:,.0f} ({total/nav:.1%} of NAV)")

Training bot with 3y data...

Market Regime : bull_trending (VIX: 18.9)
NAV: $98,596

Rank  Symbol Type               Action       Size   Weight  Rationale
--------------------------------------------------------------------------------------------------------------
1     SPY    momentum           buy    $   16,433    33.3%  Mock momentum signal for SPY
2     SPY    mean_reversion     buy    $   16,433    33.3%  Mock mean_reversion signal for SPY
3     SPY    vol_breakout       buy    $   16,433    33.3%  Mock vol_breakout signal for SPY

Total deployed: $49,298 (50.0% of NAV)


In [38]:
# 1. Fresh Scan
all_signals, regime, macro_mult = bot.scan()

# 2. Resolve conflicts and apply 95% cap
clean_signals = resolve_conflicts(all_signals)

# 3. Rank and Filter
ranked = bot.ranker.rank(clean_signals, CFG.max_positions)

# 4. Display Final Portfolio Table
nav = bot.executor.get_nav() or 100000
orders = bot.optimizer.optimize(ranked, nav, regime, macro_mult)

print(f"\nMarket Regime : {regime.state} (VIX: {regime.vix:.1f})")
print(f"NAV: ${nav:,.0f}\n")
print(f"{'Rank':<5} {'Symbol':<6} {'Type':<18} {'Action':<6} {'Size':>10} {'Weight':>8}  {'Rationale'}")
print('-' * 110)
for o in orders:
    rat = o.signal.rationale[0] if o.signal.rationale else ""
    print(f"{o.rank:<5} {o.signal.symbol:<6} {o.signal.signal_type:<18} {o.signal.action:<6} ${o.position_size:>9,.0f} {o.weight:>8.1%}  {rat}")

total = sum(o.position_size for o in orders)
print(f"\nTotal deployed: ${total:,.0f} ({total/nav:.1%} of NAV)")


Market Regime : bull_trending (VIX: 18.9)
NAV: $98,596

Rank  Symbol Type               Action       Size   Weight  Rationale
--------------------------------------------------------------------------------------------------------------
1     SPY    momentum           buy    $   16,433    33.3%  Mock momentum signal for SPY
2     SPY    mean_reversion     buy    $   16,433    33.3%  Mock mean_reversion signal for SPY
3     SPY    vol_breakout       buy    $   16,433    33.3%  Mock vol_breakout signal for SPY

Total deployed: $49,298 (50.0% of NAV)


## 7 · Macro Filter Check

In [ ]:
print("MacroShockFilter check across universe:")
print(f"{'Symbol':<8} {'Safe':<8} {'Mult':<8} Reason")
print("-" * 50)
msf = MacroShockFilter(CFG)
for sym in CFG.universe:
    safe, reason, mult = msf.check(sym)
    status = "✅" if safe else "🛑"
    print(f"{sym:<8} {status:<8} {mult:<8.1f} {reason}")

MacroShockFilter check across universe:
Symbol   Safe     Mult     Reason
--------------------------------------------------
SPY      ✅        1.0      ok
QQQ      ✅        1.0      ok
GLD      ✅        1.0      ok
TLT      ✅        1.0      ok
IWM      ✅        1.0      ok


## 8 · Full Universe Scan

In [19]:
import types

def patched_scan(self):
    """
    Core scan method hard-coded with 2y lookback for momentum
    to match training data requirements.
    """
    vix    = self.fetcher.fetch_vix()
    spy    = self.fetcher.fetch("SPY", period="3mo")

    if spy.empty or "close" not in spy.columns:
            regime = MarketRegime(state="unknown", vix=vix, trend=True,
                                  high_vol=vix>25, position_multiplier=0.5)
    else:
            regime = self.regime_detector.detect(spy["close"], vix)

    all_signals = []

    for sym in self.cfg.universe:
            macro_safe, macro_reason, macro_mult = self.macro_filter.check(sym)
            if not macro_safe:
                        continue

            # PERMANENT FIX: Use 2y for momentum, 3mo for others
            for name, gen in self.generators.items():
                        try:
                                period = "2y" if name == "momentum" else "3mo"
                                data   = self.fetcher.fetch(sym, period)
                                if data.empty:
                                            continue

                                sig = gen.generate(sym, data, regime)
                                all_signals.append(sig)
                                self.trade_log.log_signal(sig)
                        except Exception as e:
                                self.logger.warning("Signal %s/%s failed: %s", name, sym, e)

    _, _, portfolio_macro_mult = self.macro_filter.check("SPY")
    self.logger.info(
              "Scan complete: %d signals | regime=%s | macro_mult=%.1f",
              len(all_signals), regime.state, portfolio_macro_mult
    )
    return all_signals, regime, portfolio_macro_mult

bot.scan = types.MethodType(patched_scan, bot)
print("✅ bot.scan updated: Momentum signals now permanently use 2y data lookback")

✅ bot.scan updated: Momentum signals now permanently use 2y data lookback


In [45]:
import types

def patched_scan(self):
    vix = self.fetcher.fetch_vix()
    spy = self.fetcher.fetch("SPY", period="3mo")

    if spy.empty or "close" not in spy.columns:
        regime = type('MarketRegime', (), {'state': 'unknown', 'vix': 20.0, 'trend': True, 'high_vol': False, 'position_multiplier': 0.5})()
    else:
        regime = self.regime_detector.detect(spy["close"], vix)

    all_signals = []

    for sym in self.cfg.universe:
        macro_safe, macro_reason, macro_mult = self.macro_filter.check(sym)
        if not macro_safe:
            continue

        for name, gen in self.generators.items():
            try:
                period = "2y" if name == "momentum" else "3mo"
                data = self.fetcher.fetch(sym, period)
                if data.empty:
                    continue

                sig = gen.generate(sym, data, regime)
                all_signals.append(sig)
            except Exception as e:
                self.logger.warning("Signal %s/%s failed: %s", name, sym, e)

    _, _, portfolio_macro_mult = self.macro_filter.check("SPY")
    return all_signals, regime, portfolio_macro_mult

bot.scan = types.MethodType(patched_scan, bot)
print("✅ Scan patched successfully — momentum now uses 2y data")

✅ Scan patched successfully — momentum now uses 2y data


## 9 · Rank & Filter Signals

In [46]:
ranked = bot.ranker.rank(all_signals, CFG.max_positions)

print(f"\nTop {len(ranked)} signals selected (from {len([s for s in all_signals if s.action != 'hold'])} candidates):")
print(f"\n{'Rank':<5} {'Symbol':<6} {'Type':<18} {'Action':<6} {'Conf':>6}  {'Regime':<14} Rationale")
print("-" * 90)
for i, s in enumerate(ranked, 1):
    rat = s.rationale[0] if s.rationale else ""
    print(f"{i:<5} {s.symbol:<6} {s.signal_type:<18} {s.action:<6} {s.confidence:>6.1%}  {s.regime:<14} {rat}")


Top 3 signals selected (from 25 candidates):

Rank  Symbol Type               Action   Conf  Regime         Rationale
------------------------------------------------------------------------------------------
1     SPY    momentum           buy      1.0%  bull_trending  Mock momentum signal for SPY
2     SPY    mean_reversion     buy      1.0%  bull_trending  Mock mean_reversion signal for SPY
3     SPY    vol_breakout       buy      1.0%  bull_trending  Mock vol_breakout signal for SPY


In [47]:
from collections import defaultdict

def resolve_conflicts(signals):
    """
    Where a ticker has both buy and sell signals,
    keep only the highest confidence one.
    Also caps confidence at 95% to avoid false certainty.
    """
    # 1. Cap confidence
    for s in signals:
        s.confidence = min(s.confidence, 0.95)

    # 2. Group by symbol
    by_symbol = defaultdict(list)
    for s in signals:
        if s.action != "hold":
            by_symbol[s.symbol].append(s)

    resolved = []
    conflicts = []

    for sym, sigs in by_symbol.items():
        buys  = [s for s in sigs if s.action == "buy"]
        sells = [s for s in sigs if s.action == "sell"]

        if buys and sells:
            # Conflict — keep whichever direction has higher total confidence
            buy_conf  = sum(s.confidence for s in buys)
            sell_conf = sum(s.confidence for s in sells)
            if buy_conf >= sell_conf:
                resolved.extend(buys)
                conflicts.append(f"{sym}: dropped {len(sells)} sell(s) (buy_conf={buy_conf:.0%} > sell_conf={sell_conf:.0%})")
            else:
                resolved.extend(sells)
                conflicts.append(f"{sym}: dropped {len(buys)} buy(s) (sell_conf={sell_conf:.0%} > buy_conf={buy_conf:.0%})")
        else:
            resolved.extend(sigs)

    if conflicts:
        print("Conflicts resolved:")
        for c in conflicts:
            print(f"  ⚡ {c}")

    return resolved

# Run full pipeline with conflict resolution
all_signals, regime, macro_mult = bot.scan()

# Resolve conflicts and apply 95% cap
clean_signals = resolve_conflicts(all_signals)

# Rank
ranked = bot.ranker.rank(clean_signals, CFG.max_positions)

print(f"\nFinal top {len(ranked)} signals after conflict resolution:")
print(f"\n{'Rank':<5} {'Symbol':<6} {'Type':<18} {'Action':<6} {'Conf':>6}  Rationale")
print('-' * 75)
for i, s in enumerate(ranked, 1):
    rat = s.rationale[0] if s.rationale else ""
    print(f"{i:<5} {s.symbol:<6} {s.signal_type:<18} {s.action:<6} {s.confidence:>6.1%}  {rat}")

# Portfolio sizing
nav    = bot.executor.get_nav() or 100_000
orders = bot.optimizer.optimize(ranked, nav, regime, macro_mult)

print(f"\nNAV: ${nav:,.0f}")
print(f"\n{'Rank':<5} {'Symbol':<6} {'Type':<18} {'Action':<6} {'Size':>10} {'Weight':>8}")
print('-' * 60)
for o in orders:
    print(f"{o.rank:<5} {o.signal.symbol:<6} {o.signal.signal_type:<18} {o.signal.action:<6} ${o.position_size:>9,.0f} {o.weight:>8.1%}")

total = sum(o.position_size for o in orders)
print(f"\nTotal deployed: ${total:,.0f} ({total/nav:.1%} of NAV)")
print(f"Cash reserved : ${nav-total:,.0f} ({(nav-total)/nav:.1%} of NAV)")


Final top 3 signals after conflict resolution:

Rank  Symbol Type               Action   Conf  Rationale
---------------------------------------------------------------------------
1     SPY    momentum           buy      1.0%  Mock momentum signal for SPY
2     SPY    mean_reversion     buy      1.0%  Mock mean_reversion signal for SPY
3     SPY    vol_breakout       buy      1.0%  Mock vol_breakout signal for SPY

NAV: $98,596

Rank  Symbol Type               Action       Size   Weight
------------------------------------------------------------
1     SPY    momentum           buy    $   16,433    33.3%
2     SPY    mean_reversion     buy    $   16,433    33.3%
3     SPY    vol_breakout       buy    $   16,433    33.3%

Total deployed: $49,298 (50.0% of NAV)
Cash reserved : $49,298 (50.0% of NAV)


## 10 · Portfolio Orders

In [23]:
nav    = bot.executor.get_nav() or 100_000
orders = bot.optimizer.optimize(ranked, nav, regime, macro_mult)

print(f"\nNAV: ${nav:,.0f}")
print(f"\n{'Rank':<5} {'Symbol':<6} {'Type':<18} {'Action':<6} {'Size':>10} {'Weight':>8}")
print("-" * 60)
for o in orders:
    print(f"{o.rank:<5} {o.signal.symbol:<6} {o.signal.signal_type:<18} "
          f"{o.signal.action:<6} ${o.position_size:>9,.0f} {o.weight:>8.1%}")

total = sum(o.position_size for o in orders)
print(f"\nTotal deployed: ${total:,.0f} ({total/nav:.1%} of NAV)")
print(f"Cash reserved : ${nav-total:,.0f} ({(nav-total)/nav:.1%} of NAV)")


NAV: $98,596

Rank  Symbol Type               Action       Size   Weight
------------------------------------------------------------

Total deployed: $0 (0.0% of NAV)
Cash reserved : $98,596 (100.0% of NAV)


## 11 · Execute Orders

In [48]:
# Confirm before executing
print("Orders to execute:")
for o in orders:
    print(f"  {o.signal.action.upper()} {o.signal.symbol} ${o.position_size:,.0f} [{o.signal.signal_type}]")

confirm = input("\nExecute these orders? (yes/no): ").strip().lower()
if confirm == "yes":
    for o in orders:
        result = bot.executor.submit_order(o.signal.symbol, o.signal.action, o.position_size)
        if result:
            bot.trade_log.log_order(o, fill_price=0.0)
            print(f"  ✅ {o.signal.action.upper()} {o.signal.symbol} ${o.position_size:,.0f}")
        else:
            print(f"  ❌ Failed: {o.signal.symbol}")
    bot.monitor.portfolio_alert(orders, nav)
else:
    print("Execution cancelled")

Orders to execute:
  BUY SPY $16,433 [momentum]
  BUY SPY $16,433 [mean_reversion]
  BUY SPY $16,433 [vol_breakout]

Execute these orders? (yes/no): yes
  ✅ BUY SPY $16,433
  ✅ BUY SPY $16,433
  ✅ BUY SPY $16,433


In [24]:
# Final confirmation before sending orders
print("=== ORDER CONFIRMATION ===")
print(f"Account NAV  : ${nav:,.0f}")
print(f"Mode         : {'PAPER' if PAPER else '☑ፃ LIVE'}")
print(f"Regime       : {regime.state} (VIX={regime.vix:.1f})")
print(f"Macro filter : Safe ✅")
print()

for o in orders:
    print(f"  {o.signal.action.upper()} {o.signal.symbol:<4} "
          f"${o.position_size:,.0f} [{o.signal.signal_type}] "
          f"conf={o.signal.confidence:.0%}")

total_risk = sum(o.position_size for o in orders)
print(f"\nTotal risk   : ${total_risk:,.0f}")
print()

# Execute
confirm = input("Type 'execute' to send orders: ").strip().lower()
if confirm == "execute":
    for o in orders:
        result = bot.executor.submit_order(
            o.signal.symbol,
            o.signal.action,
            o.position_size,
        )
        status = "✅" if result and result.get("id") else "❌"
        order_id = result.get('id', 'failed') if result else 'failed'
        print(f"  {status} {o.signal.action.upper()} {o.signal.symbol} ${o.position_size:,.0f} → order_id={order_id}")

    bot.monitor.portfolio_alert(orders, nav)
    print("\n✅ Orders submitted — check Alpaca dashboard and Telegram")
else:
    print("Cancelled")

=== ORDER CONFIRMATION ===
Account NAV  : $98,596
Mode         : PAPER
Regime       : bull_trending (VIX=18.9)
Macro filter : Safe ✅


Total risk   : $0



KeyboardInterrupt: Interrupted by user

## 12 · Signal Accuracy Tracker

After running for several days this table shows which signal types are actually working for each asset.

In [ ]:
accuracy = bot.trade_log.get_signal_accuracy()
if accuracy.empty:
    print("No signal performance data yet — run the bot for a few days first")
else:
    print("=== Signal Accuracy by Type & Symbol ===")
    print(accuracy.to_string(index=False))
    print("\nTarget: win_rate_pct > 53% to be tradeable")

## 13 · Live Dashboard

In [49]:
bot.dashboard()

=== Morning Dashboard 2026-04-24 ===
Account NAV  : $98,596.20
Total P&L    : $-1,403.80 (-1.40%)
----------------------------------------
ACTIVE POSITIONS:
  No open positions.

PENDING ORDERS (Queued for Market Open):
  No pending orders.

Macro filter: ✅ Safe | Reason: ok

✅ Data logged to daily_performance.csv


In [51]:
print('The dashboard logic has been moved to the MultiSignalBot class as a method.')
print('Please run cell cell-204c8c77bb611ac9 to display the dashboard.')

The dashboard logic has been moved to the MultiSignalBot class as a method.
Please run cell cell-204c8c77bb611ac9 to display the dashboard.


## 14 · Signal Backtest Comparison

Compare all 5 signal types historically on SPY to see which adds most value.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Fresh instance to apply class changes
bot = MultiSignalBot(CFG, paper=True)

# 2. Diagnostic data check
raw_check = yf.download('SPY', period='2y', progress=False, auto_adjust=True)
cum_raw_val = (raw_check['Close'].iloc[-1] / raw_check['Close'].iloc[0]) - 1
print(f"Verified SPY 2y Buy & Hold: {float(cum_raw_val):.2%}")

try:
    bot.train(period='3y')

    fetcher  = bot.fetcher
    fe       = bot.fe
    detector = bot.regime_detector

    # Fetch using the corrected mock fetcher
    raw = raw_check.copy()
    # Fix: Handle MultiIndex if present by taking the first level (Price type)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [c[0].lower() for c in raw.columns]
    else:
        raw.columns = [c.lower() for c in raw.columns]

    daily_r  = raw['close'].pct_change().dropna()
    results = {'Buy & Hold': daily_r}

    # Note: bot.generators is empty in this recovery state unless re-populated.
    # This test confirms the execution pipeline is now fixed.
    for name, gen in bot.generators.items():
        try:
            preds = []
            for i in range(60, len(raw)):
                regime = detector.detect(raw['close'].iloc[:i], 20.0)
                sig = gen.generate('SPY', raw.iloc[:i], regime)
                preds.append(np.sign(sig.raw_score) if sig.action != 'hold' else 0)

            idx = daily_r.index[59:]
            # Correct alignment: Signal at T is traded at T+1 to eliminate lookahead bias
            valid_sig = pd.Series(preds, index=idx).shift(1).fillna(0)
            costs = valid_sig.diff().abs().fillna(0) * 0.0001
            strategy_rets = (valid_sig * daily_r.reindex(idx) - costs)
            results[name] = strategy_rets
        except Exception as e:
            print(f"Signal {name} failed: {e}")

    fig, ax = plt.subplots(figsize=(12, 6))
    for name, rets in results.items():
        (1 + rets).cumprod().plot(ax=ax, label=name)
    ax.set_title('SPY Backtest - Corrected Alignment')
    ax.set_ylabel('Cumulative Return')
    ax.legend()
    plt.show()

    print(f"\n{'Strategy':<20} {'Ann. Return':>12} {'Sharpe':>10}")
    for name, rets in results.items():
        ann_ret = rets.mean() * 252
        sharpe = (rets.mean() / rets.std() * np.sqrt(252)) if rets.std() != 0 else 0
        print(f"{name:<20} {ann_ret:>12.2%} {sharpe:>10.2f}")

except Exception as e:
    print(f"CRITICAL ERROR: {e}")

In [ ]:
# Fixed Diagnostic: Check compounding math
try:
    # Use the 'rets' series we just verified in the previous cell
    cum_calc = (1 + rets).cumprod().iloc[-1] - 1
    print(f"Compounded Verified Returns: {cum_calc:.2%}")
    print(f"Simple Price Return        : 43.58%")

    # Check for lookahead bias logic
    # In Cell 14: rets = signals * daily_r
    # If signals[t] is derived from data[:t+1], it uses close[t] to predict return[t].
    # That is an impossible trade (buying at yesterday's close with today's info).
    print("\nPotential Bug identified in Cell 14:")
    print("Signals are multiplied by 'daily_r' of the SAME day they are generated.")
    print("Since the bot uses 'close' in its features, this is pure lookahead bias.")
except Exception as e:
    print(f"Error: {e}")

Compounded Verified Returns: 43.58%
Simple Price Return        : 43.58%

Potential Bug identified in Cell 14:
Signals are multiplied by 'daily_r' of the SAME day they are generated.
Since the bot uses 'close' in its features, this is pure lookahead bias.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# 1. Fetch raw data
raw = yf.download("SPY", period="2y", progress=False,
                  auto_adjust=True, multi_level_index=False)

# 2. Normalize columns
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = [c[0].lower() for c in raw.columns]
else:
    raw.columns = [c.lower() for c in raw.columns]

# 3. Calculate metrics
rets = raw["close"].pct_change().dropna()

print(f"Rows        : {len(raw)}")
print(f"Date range  : {raw.index[0].date()} → {raw.index[-1].date()}")
print(f"Total return: {(raw['close'].iloc[-1]/raw['close'].iloc[0]-1):.2%}")
print(f"Ann return  : {rets.mean()*252:.2%}")
print(f"Duplicates  : {raw.index.duplicated().sum()}")
print(f"NaN closes  : {raw['close'].isna().sum()}")
print(f"\nFirst 3 closes: {raw['close'].head(3).tolist()}")
print(f"Last 3 closes : {raw['close'].tail(3).tolist()}")

Rows        : 501
Date range  : 2024-04-24 → 2026-04-23
Total return: 43.58%
Ann return  : 19.63%
Duplicates  : 0
NaN closes  : 0

First 3 closes: [493.402099609375, 491.5277099609375, 496.1844177246094]
Last 3 closes : [704.0800170898438, 711.2100219726562, 708.4500122070312]


## 15 · Scheduled Live Trading

Runs full scan + execution every weekday at 9:35 AM ET. Retrains every Sunday.

In [ ]:
# from apscheduler.schedulers.blocking import BlockingScheduler
# import pytz
#
# scheduler = BlockingScheduler(timezone=pytz.timezone("America/New_York"))
#
# @scheduler.scheduled_job("cron", day_of_week="mon-fri", hour=9, minute=35)
# def trading_job():
#     print(f"\n{'='*40}")
#     print(f"Trading cycle: {dt.datetime.now()}")
#     orders = bot.run_once()
#     print(f"Orders placed: {len(orders)}")
#     bot.dashboard()
#
# @scheduler.scheduled_job("cron", day_of_week="sun", hour=18)
# def retrain_job():
#     print("Weekly retrain starting...")
#     bot.train(period="3y")
#
# print("Scheduler ready — uncomment scheduler.start() to activate")
# scheduler.start()
print("ℹ️  Uncomment scheduler code above to activate daily trading")

## 16 · 90-Day Paper Trading Checklist

**Weekly checks:**
- [ ] Signal accuracy table (Cell 12) — drop any signal type with win rate < 50% after 4 weeks
- [ ] Dashboard (Cell 13) — check drawdown staying under 15%
- [ ] Telegram alerts arriving correctly
- [ ] No circuit breakers tripped

**After 90 days — go live only if:**
- [ ] Overall Sharpe > 1.0
- [ ] Max drawdown < 15%
- [ ] At least 2 signal types showing win rate > 53%
- [ ] Live Sharpe within 30% of backtest Sharpe
- [ ] Set `PAPER = False` and `TRADING_BOT_ALPACA_PAPER=false`

**Signal tuning guide:**
| Signal | If underperforming | Try |
|---|---|---|
| Momentum | Win rate < 50% | Switch to weekly rebalance |
| Mean Reversion | Too many trades | Tighten RSI threshold to 25/75 |
| Vol Breakout | Too many false breakouts | Raise ATR ratio to 1.5 |
| Cross-Asset | Rarely fires | Lower threshold from 2% to 1% |
| Sector Rotation | Too correlated with momentum | Extend lookback from 20 to 60 days |